### The OpenAI Agents SDK Docs

https://openai.github.io/openai-agents-python/

In [ ]:
!pip install openai-agents

In [ ]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent

from agents import Agent, Runner, trace, function_tool, SQLiteSession

load_dotenv(override=True)


## Part 1: A simple agent

In [ ]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [ ]:
# Here is the final output

print(result.final_output)

In [ ]:
# Here is the detail of the LLM calls

result.to_input_list()

### Adding Observability with a trace (see the consumed time in openai dashboard)

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

### Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# use function_tool decorartor

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
push_tool.description

In [ ]:
# the notifier agent

notifier = Agent(name="Notifier",
                 model="gpt-5.4-mini", 
                 instructions="You notify the user upon request", 
                 tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


### look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (add memory)

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

### call without any memory

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

### memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

### memory approach 2 - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory
